In [1]:
# Top cell: install the libraries this notebook uses.
# If they are already installed, pip will simply say so.
%pip install -q pandas matplotlib numpy scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'
CH = ROOT / 'charts'
CH.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 150})
print('Paths ready:', PROC.exists(), CH.exists())

Paths ready: True True


# 5. Two original checks: warming pace, and cost versus frequency

This notebook adds two checks that are not simply copied from a source chart.

Check 1, warming pace: for each decade, take the decade average minus the previous decade average. That change is the pace for that decade. A larger positive change means faster warming that decade.

Check 2, cost versus frequency: for disasters, take total cost divided by event count for each decade. That is the average cost per event. If total cost rises only because there are more events, the average per event will stay flat. If each event is getting costlier, the average will rise.

Both checks use only the processed tables from notebooks 1 and 3, so every input can be traced.

In [3]:
d = pd.read_csv(PROC / 'decade_means.csv')
print(d.to_string(index=False))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(d.dropna(subset=['change_from_previous_decade_c'])['decade'], d.dropna(subset=['change_from_previous_decade_c'])['change_from_previous_decade_c'], color='#d62728')
ax.axhline(0, color='grey', linewidth=0.8)
ax.set_title('How much the decade average moved, compared with the decade before')
ax.set_xlabel('Decade')
ax.set_ylabel('Change in degrees Celsius')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
fig.savefig(CH / '03_decade_change_pace.png', bbox_inches='tight')
plt.show()

                decade  mean_anomaly_c  years_included  change_from_previous_decade_c
                 1880s          -0.209              10                            NaN
                 1890s          -0.235              10                         -0.026
                 1900s          -0.313              10                         -0.078
                 1910s          -0.326              10                         -0.013
                 1920s          -0.238              10                          0.088
                 1930s          -0.122              10                          0.116
                 1940s           0.042              10                          0.164
                 1950s          -0.047              10                         -0.089
                 1960s          -0.031              10                          0.016
                 1970s           0.035              10                          0.066
                 1980s           0.247              10

In [4]:
g = pd.read_csv(PROC / 'global_annual.csv')
for a, b in [(1880, 2023), (1970, 2023), (1980, 2023), (2000, 2023)]:
    s = g[(g['year'] >= a) & (g['year'] <= b)]
    slope_per_decade = np.polyfit(s['year'], s['annual_anomaly_c'], 1)[0] * 10
    print(f'Linear trend {a} to {b}: {slope_per_decade:.3f} degrees Celsius per decade')
print('Reading: the trend since 1980 is steeper than the trend since 1880. Warming sped up.')

Linear trend 1880 to 2023: 0.079 degrees Celsius per decade
Linear trend 1970 to 2023: 0.193 degrees Celsius per decade
Linear trend 1980 to 2023: 0.195 degrees Celsius per decade
Linear trend 2000 to 2023: 0.235 degrees Celsius per decade
Reading: the trend since 1980 is steeper than the trend since 1880. Warming sped up.


In [5]:
bd = pd.read_csv(PROC / 'disasters_by_decade.csv')
print(bd.to_string(index=False))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(bd['decade'], bd['mean_cost_per_event_billion_usd'], color='#17becf')
ax.set_title('Average cost per billion-dollar event, by decade')
ax.set_xlabel('Decade (2020s covers 2020-2024 only)')
ax.set_ylabel('Billion US dollars per event, CPI adjusted')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
fig.savefig(CH / '12_cost_per_event_by_decade.png', bbox_inches='tight')
plt.show()

           decade  event_count  total_cost_cpi_billion_usd  total_deaths  mean_cost_per_event_billion_usd
            1980s           33                    219.6485          3017                         6.656015
            1990s           57                    335.1072          3075                         5.879074
            2000s           67                    621.9355          3102                         9.282619
            2010s          131                    994.3862          5227                         7.590734
2020s (2020-2024)          115                    746.5291          2520                         6.491557


In [6]:
g2 = g.copy()
g2['rolling_10yr'] = g2['annual_anomaly_c'].rolling(10).mean()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(g2['year'], g2['annual_anomaly_c'], color='#bbbbbb', linewidth=0.8, label='Single year')
ax.plot(g2['year'], g2['rolling_10yr'], color='#1f77b4', linewidth=2, label='10 year rolling mean')
ax.axhline(0, color='grey', linewidth=0.8); ax.legend()
ax.set_title('Single years jump around. The 10 year mean shows the trend.')
ax.set_xlabel('Year')
ax.set_ylabel('Anomaly in degrees Celsius')
fig.savefig(CH / '04_rolling_10yr_mean.png', bbox_inches='tight')
plt.show()

## What the two checks find

Warming pace: the decade average rose by 0.212 degrees from the 1970s to the 1980s, by 0.204 from the 1990s to the 2000s, and by 0.217 from the 2000s to the 2010s. The linear trend is 0.079 degrees per decade since 1880, but 0.195 per decade since 1980 and 0.235 per decade since 2000. In plain words, recent decades warmed faster than the long run average.

Cost versus frequency: the average cost per billion-dollar event was 6.66 billion dollars in the 1980s, 5.88 in the 1990s, 9.28 in the 2000s, 7.59 in the 2010s, and 6.49 in 2020 to 2024. It bounces around, it does not climb steadily. Meanwhile the event count climbed from 33 to 131 to 115 in five years. So the rise in total disaster cost in this dataset is driven mainly by more events crossing the line, not by a steady rise in the average cost of each event. That conclusion is limited to this dataset and its threshold. It does not say anything about smaller events, which this dataset excludes by design.